# Pedestrian Volume Prediction using Nested MERF-XGB

This notebook demonstrates the official implementation of the **Mixed-Effects Regression Forest with XGBoost Fixed Effects (MERF-XGB)** modeling framework. The code predicts pedestrian Monthly Average Daily Traffic (MADT) by accounting for both spatial and longitudinal clustering.

### Pipeline Overview:
1. **Data Preprocessing & Nested Clustering**: Setting up spatial-temporal grouping variables.
2. **Strategy Comparison**: Comparing Local, Global, Global + Cluster Features, and MERF-XGB models on the same footing.
3. **Hyperparameter Tuning**: Finding the best boosting hyperparameters using Group-K-Fold cross-validation.
4. **Final Model Evaluation**: Assessing the final tuned model.
5. **SHAP-based Feature Attribution**: Quantifying global and location-specific feature importance.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shap
import warnings
from itertools import product
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from merf import MERF
from xgboost import XGBRegressor

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid')


## 1. Data Preparation
We load the pedestrian counter dataset. If the original `MDOT_Edited_Data.xlsx` dataset is not found in the directory (e.g. when run from a public repository), the pipeline automatically falls back to `MDOT_Synthetic_Data.xlsx` which has identical structure and statistics.


In [ ]:
# 1. Load dataset (original or synthetic)
file_path = 'MDOT_Edited_Data.xlsx'
if not os.path.exists(file_path):
    file_path = 'MDOT_Synthetic_Data.xlsx'
    print('Original dataset not found. Using MDOT_Synthetic_Data.xlsx for reproduction...')
else:
    print('Using original MDOT_Edited_Data.xlsx dataset.')

df = pd.read_excel(file_path, sheet_name='Pedestrian Counters')

# 2. Define features, target, and clustering levels
feature_cols = [
    'Strava_MADT', 'bik_den', 'slope', 'distcolleg',
    'bik_pct', 'ret_area', 'afam', 'prec',
    'hh_den', 'hum', 'med_age'
]
target_col = 'Pedestrian_Count_Average_MADT'
spatial_col = 'location'
temporal_col = 'Year'

# 3. Create nested cluster column representing Location x Year
df['nested_cluster'] = df[spatial_col].astype(str) + '_' + df[temporal_col].astype(str)

# 4. Clean missing values and split into variables
df_clean = df.dropna(subset=feature_cols + [target_col, 'nested_cluster'])

X = df_clean[feature_cols]
y = df_clean[target_col]
clusters = df_clean['nested_cluster']

# 5. Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
clusters_train = clusters.loc[X_train.index]
clusters_test = clusters.loc[X_test.index]

print(f'Data preparation complete. Total observations: {len(df_clean)}')
print(f'Train set: {len(X_train)} samples, Test set: {len(X_test)} samples')
print(f'Unique nested site-year clusters: {len(clusters.unique())}')


## 2. Strategy Comparison (Baseline)
We compare the four modeling strategies on the same footing using baseline, default hyperparameters for the boosting component (`max_depth=4`, `learning_rate=0.1`, `n_estimators=50`):
1. **Local Models**: Independent XGBoost models fit per site-year cluster.
2. **Global Model**: A single standard XGBoost model trained globally.
3. **Global + Cluster Features**: A global XGBoost model that includes one-hot encoded site-year cluster dummies.
4. **MERF-XGB**: Our proposed mixed-effects model incorporating site-year random intercepts.


In [ ]:
# Define baseline parameters
gpu_params = {
    'n_estimators': 50,
    'learning_rate': 0.1,
    'max_depth': 4,
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1
}

def get_metrics(y_true, y_pred, model_name, dataset='Test'):
    return {
        'Strategy': model_name,
        'Dataset': dataset,
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred)
    }

results_list = []

# --- Strategy 1: Local Models ---
print('Running Strategy 1: Local Models...')
def local_model_predict(X_ref, y_ref, clusters_ref, X_target, clusters_target):
    y_pred = np.zeros(len(X_target))
    target_indices = X_target.index
    unique_clusters = clusters_target.unique()
    for cluster in unique_clusters:
        train_idx = clusters_ref[clusters_ref == cluster].index
        test_idx = clusters_target[clusters_target == cluster].index
        if len(train_idx) > 3:
            model = XGBRegressor(**gpu_params)
            model.fit(X_ref.loc[train_idx], y_ref.loc[train_idx])
            preds = model.predict(X_target.loc[test_idx])
        else:
            preds = [y_ref.mean()] * len(test_idx)
        for i, idx in enumerate(test_idx):
            pos = np.where(target_indices == idx)[0][0]
            y_pred[pos] = preds[i]
    return y_pred

y_pred_local_train = local_model_predict(X_train, y_train, clusters_train, X_train, clusters_train)
y_pred_local_test = local_model_predict(X_train, y_train, clusters_train, X_test, clusters_test)
results_list.append(get_metrics(y_train, y_pred_local_train, '1. Local Models', 'Train'))
results_list.append(get_metrics(y_test, y_pred_local_test, '1. Local Models', 'Test'))

# --- Strategy 2: Global Model ---
print('Running Strategy 2: Global Model...')
model_global = XGBRegressor(**gpu_params)
model_global.fit(X_train, y_train)
results_list.append(get_metrics(y_train, model_global.predict(X_train), '2. Global Model', 'Train'))
results_list.append(get_metrics(y_test, model_global.predict(X_test), '2. Global Model', 'Test'))

# --- Strategy 3: Global + Cluster Feature ---
print('Running Strategy 3: Global + Cluster Feature...')
X_with_cluster = pd.get_dummies(df_clean[feature_cols + ['nested_cluster']], columns=['nested_cluster'], drop_first=True)
X_train_OHE = X_with_cluster.loc[X_train.index]
X_test_OHE = X_with_cluster.loc[X_test.index]
X_train_OHE, X_test_OHE = X_train_OHE.align(X_test_OHE, join='outer', axis=1, fill_value=False)

model_ohe = XGBRegressor(**gpu_params)
model_ohe.fit(X_train_OHE, y_train)
results_list.append(get_metrics(y_train, model_ohe.predict(X_train_OHE), '3. Global + Cluster Feature', 'Train'))
results_list.append(get_metrics(y_test, model_ohe.predict(X_test_OHE), '3. Global + Cluster Feature', 'Test'))

# --- Strategy 4: MERF-XGB (Baseline) ---
print('Running Strategy 4: MERF (Mixed-Effects)...')
merf_baseline = MERF(fixed_effects_model=XGBRegressor(**gpu_params), max_iterations=20)
Z_train = np.ones((len(X_train), 1))
Z_test = np.ones((len(X_test), 1))
merf_baseline.fit(X_train, Z_train, clusters_train, y_train)
results_list.append(get_metrics(y_train, merf_baseline.predict(X_train, Z_train, clusters_train), '4. MERF', 'Train'))
results_list.append(get_metrics(y_test, merf_baseline.predict(X_test, Z_test, clusters_test), '4. MERF', 'Test'))

results_df = pd.DataFrame(results_list)
print('\nStrategy Comparison Table (Untuned):')
print(results_df.round(4))


### Plot Baseline RMSE Comparison
We visualize the performance (RMSE) across the baseline strategies.


In [ ]:
plt.figure(figsize=(12, 6))
plot = sns.barplot(x='Strategy', y='RMSE', hue='Dataset', data=results_df, palette=['#345e7d', '#4ba87d'])
plt.title('Baseline Modeling Strategy Comparison (RMSE - Lower is Better)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('RMSE (Pedestrian MADT)', fontsize=12)
plt.xlabel('Modeling Strategy', fontsize=12)
for p in plot.patches:
    if p.get_height() > 0:
        plot.annotate(format(p.get_height(), '.1f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 8),
                      textcoords='offset points', fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.5)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()


## 3. Hyperparameter Tuning for MERF-XGB
To optimize the predictive accuracy of the MERF-XGB model, we perform a grid search over key XGBoost parameters using a cluster-aware **Group-K-Fold cross-validation** strategy. Group-K-Fold ensures that the nested site-year clusters are kept intact during splitting, preventing leakage and providing a realistic generalizability score.


In [ ]:
print('Starting Grid Search with GroupKFold...')
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

best_score = np.inf
best_params = None

for params in product(*param_grid.values()):
    param_dict = dict(zip(param_grid.keys(), params))
    cv_rmse = []

    for train_idx, val_idx in gkf.split(X_train, y_train, groups=clusters_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        clusters_tr, clusters_val = clusters_train.iloc[train_idx], clusters_train.iloc[val_idx]

        xgb_model = XGBRegressor(**param_dict, tree_method='hist', random_state=42, n_jobs=-1)
        merf_model = MERF(fixed_effects_model=xgb_model, max_iterations=15)

        Z_tr = np.ones((len(X_tr), 1))
        Z_val = np.ones((len(X_val), 1))

        merf_model.fit(X_tr, Z_tr, clusters_tr, y_tr)
        y_val_pred = merf_model.predict(X_val, Z_val, clusters_val)
        cv_rmse.append(np.sqrt(mean_squared_error(y_val, y_val_pred)))

    mean_cv_rmse = np.mean(cv_rmse)
    if mean_cv_rmse < best_score:
        best_score = mean_cv_rmse
        best_params = param_dict

print(f'Best Cross-Validation RMSE: {best_score:.4f}')
print(f'Optimized Hyperparameters: {best_params}')


## 4. Final Model Training and Evaluation
We train the final MERF-XGB model on the full training dataset using the optimized hyperparameters.


In [ ]:
print('Training final MERF-XGB model...')
final_xgb = XGBRegressor(**best_params, tree_method='hist', random_state=42, n_jobs=-1)
final_merf = MERF(fixed_effects_model=final_xgb, max_iterations=30)

Z_train = np.ones((len(X_train), 1))
Z_test = np.ones((len(X_test), 1))

final_merf.fit(X_train, Z_train, clusters_train, y_train)

y_train_pred = final_merf.predict(X_train, Z_train, clusters_train)
y_test_pred = final_merf.predict(X_test, Z_test, clusters_test)

def rmse_percentage(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return (rmse / np.mean(y_true)) * 100

train_metrics = {
    'RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
    'MAE': mean_absolute_error(y_train, y_train_pred),
    'R2': r2_score(y_train, y_train_pred),
    'RMSE%': rmse_percentage(y_train, y_train_pred)
}

test_metrics = {
    'RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
    'MAE': mean_absolute_error(y_test, y_test_pred),
    'R2': r2_score(y_test, y_test_pred),
    'RMSE%': rmse_percentage(y_test, y_test_pred)
}

final_results_df = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R2', 'RMSE%'],
    'Train Set': [train_metrics['RMSE'], train_metrics['MAE'], train_metrics['R2'], train_metrics['RMSE%']],
    'Test Set': [test_metrics['RMSE'], test_metrics['MAE'], test_metrics['R2'], test_metrics['RMSE%']]
})

print('\nFinal Model Performance Table:')
print(final_results_df.round(4))


### Predicted vs. Actual and Residual Plots
We plot the predicted versus actual pedestrian MADT values along with the residual plot to assess fit and diagnose potential heteroscedasticity.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. Actual vs Predicted
sns.scatterplot(x=y_test, y=y_test_pred, alpha=0.7, ax=axes[0], color='#345e7d', s=50)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Pedestrian Count (MADT)', fontsize=12)
axes[0].set_ylabel('Predicted Pedestrian Count (MADT)', fontsize=12)
axes[0].set_title('Predicted vs. Actual Pedestrian Count', fontsize=14, fontweight='bold')

# 2. Residual Plot
residuals = y_test - y_test_pred
sns.scatterplot(x=y_test_pred, y=residuals, alpha=0.7, ax=axes[1], color='#4ba87d', s=50)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Pedestrian Count (MADT)', fontsize=12)
axes[1].set_ylabel('Residuals (Actual - Predicted)', fontsize=12)
axes[1].set_title('Residuals vs. Predicted Volume', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. SHAP Feature Attribution Analysis
To understand the relative contribution of each environmental and demographic variable in our non-linear fixed-effects component, we perform SHAP (SHapley Additive exPlanations) analysis. We use professional labels for features and visualize their importances globally and across spatial location clusters.


In [ ]:
print('Running SHAP Analysis (using KernelExplainer for MERF-XGB)...')
feature_mapping = {
    'Strava_MADT': 'Strava MADT',
    'bik_den': 'Bike Usage Density',
    'slope': 'Slope',
    'distcolleg': 'Distance to College',
    'bik_pct': 'Bike Usage Percentage',
    'ret_area': 'Retail Area',
    'afam': 'African American Pop',
    'prec': 'Precipitation',
    'hh_den': 'Household Density',
    'hum': 'Humidity',
    'med_age': 'Median Age'
}

# Sample test set for SHAP execution speed (SHAP KernelExplainer is perturbation-based)
n_test_sample = min(100, len(X_test))
X_test_sampled = X_test.sample(n=n_test_sample, random_state=42)
location_sampled = df_clean.loc[X_test_sampled.index, 'location']

def _make_cluster_series_for_x(n_samples, cluster_source):
    if cluster_source is None:
        return pd.Series(np.zeros(n_samples, dtype=int))
    cs = np.asarray(cluster_source).ravel()
    if len(cs) == 0:
        return pd.Series(np.zeros(n_samples, dtype=int))
    reps = int(np.ceil(n_samples / len(cs)))
    tiled = np.tile(cs, reps)[:n_samples]
    return pd.Series(tiled)

bg_sample = X_train.sample(n=min(100, len(X_train)), random_state=0)

def merf_predict_wrapper(x):
    df_x = pd.DataFrame(x, columns=X_train.columns)
    Z_x = np.ones((df_x.shape[0], 1))
    clusters_for_x = _make_cluster_series_for_x(df_x.shape[0], clusters_test)
    preds = final_merf.predict(df_x, Z_x, clusters_for_x)
    return np.asarray(preds).ravel()

explainer = shap.KernelExplainer(merf_predict_wrapper, bg_sample)
shap_values = explainer.shap_values(X_test_sampled)

# 1. Beeswarm Plot (Professional Names)
X_test_sampled_renamed = X_test_sampled.rename(columns=feature_mapping)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sampled_renamed, show=False)
plt.title('SHAP Values for Pedestrian MADT (Impact on Model Predictions)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# 2. Location-Based Feature Importance Plot
shap_array = np.array(shap_values)
if shap_array.ndim == 1:
    shap_array = shap_array.reshape(-1, len(X_train.columns))

shap_df = pd.DataFrame(shap_array, columns=X_test_sampled.columns)
shap_df['location'] = location_sampled.values

shap_abs_melted = shap_df.melt(id_vars='location', var_name='feature', value_name='val')
shap_abs_melted['val'] = shap_abs_melted['val'].abs()
shap_abs_melted['feature'] = shap_abs_melted['feature'].map(feature_mapping)

pivot_shap = shap_abs_melted.groupby(['feature', 'location'])['val'].mean().unstack()
feature_order = shap_abs_melted.groupby('feature')['val'].mean().sort_values(ascending=False).index
pivot_shap = pivot_shap.loc[feature_order]

pivot_shap.plot(kind='bar', figsize=(15, 7), colormap='viridis', edgecolor='black')
plt.title('Feature Importance by Location (Nested MERF-XGB)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Mean Absolute SHAP Value (MADT)', fontsize=12)
plt.xlabel('Feature', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.legend(title='Location Cluster')
plt.tight_layout()
plt.show()
